# 🧠 Mental Health Text Classifier — LSTM

This notebook builds an end-to-end NLP pipeline to classify mental health-related text into 7 categories:
**Normal, Depression, Suicidal, Anxiety, Bipolar, Stress, Personality Disorder**

**Pipeline overview:**
1. Load & explore the dataset
2. Clean & preprocess text (lowercase, remove punctuation, stopwords, lemmatize)
3. Tokenize and pad sequences
4. Encode labels
5. Build and train a stacked LSTM model
6. Save the model and artifacts for inference

---
## 1. Import Libraries

In [39]:
import pandas as pd

---
## 2. Load the Dataset

In [ ]:
# Load the raw dataset
# encoding='latin-1' handles special characters in the text
# on_bad_lines='skip' ignores any malformed rows
df = pd.read_csv(r"Combined Data.csv", encoding='latin-1' , engine='python' , on_bad_lines='skip')

: 

In [ ]:
# Preview the dataset
df

: 

---
## 3. Data Cleaning

In [ ]:
# Drop the unnamed index column that was saved from a previous export
df.drop(['Unnamed: 0'], axis=1, inplace=True)
df

: 

In [ ]:
# Check for missing values in each column
df.isnull().sum()

: 

In [ ]:
# Remove all rows that contain null values, then verify no nulls remain
df.dropna(inplace=True)
df.isnull().sum()

: 

---
## 4. Text Preprocessing

Each sentence goes through the following steps:
- Lowercase conversion
- Remove punctuation and numbers using regex
- Strip extra whitespace
- Tokenize words using NLTK
- Remove English stopwords
- Lemmatize each word to its root form

In [ ]:
# Import NLP libraries for text preprocessing
from nltk.corpus import stopwords
import re
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize


: 

In [ ]:
# Load the English stopwords list (common words like 'the', 'is', 'in' that carry no meaning)
stopwords = stopwords.words('english')
stopwords

: 

In [ ]:
def preprocess_text(text : str):
    # Convert to lowercase so 'Anxiety' and 'anxiety' are treated the same
    text = text.lower()
    
    # Remove everything that is not a letter (punctuation, numbers, symbols)
    text = re.sub('[^a-zA-Z]', " ", text)   

    # Collapse multiple spaces into a single space
    text = re.sub(r'\s+', " ", text)
  
    # Tokenize the sentence into individual words
    words = word_tokenize(text)

    # Remove stopwords — they don't contribute to the mental health classification
    words = [i for i in words if i not in stopwords]
    
    # Lemmatize: reduce each word to its base form (e.g. 'feeling' → 'feel')
    lema = WordNetLemmatizer()
    words_lemma = [lema.lemmatize(word) for word in words] 
    
    return " ".join(words_lemma)


: 

In [ ]:
# Apply the preprocessing function to every statement in the dataset
# Results are stored in a new column 'clean_sentences'
df['clean_sentences'] = df['statement'].apply(lambda x: preprocess_text(x))

: 

In [ ]:
# Preview the dataset with the new cleaned text column
df

: 

---
## 5. Exploratory Analysis

In [ ]:
# Find the maximum sentence length in the dataset (in words)
# This helps us decide the padding length for the model
max_len = 0

for i in df['statement']:
    max_len = max(max_len, len(i.split()))

print(max_len)

: 

In [ ]:
# Same calculation using a more concise one-liner
max_len = max(len(i.split()) for i in df['statement'])
print(max_len)

: 

In [ ]:
# Print the actual longest sentence in the dataset
max_sentence = max(df['statement'], key=lambda x: len(x.split()))

print(max_sentence)

: 

In [ ]:
# Confirm the word count of the longest sentence
# Note: the max is 6300 words but most sentences are well under 100 — we use maxlen=100 for padding
len(max_sentence.split())

: 

In [ ]:
# Check the distribution of labels in the dataset
# This shows class imbalance — Normal and Depression have far more samples than Personality Disorder
df['status'].value_counts()

: 

---
## 6. Filter Valid Labels & Prepare Features

In [ ]:
# Keep only the 7 target mental health categories
# This removes any noisy or mislabeled rows that don't belong to our target classes
valid_labels = [
    'Normal',
    'Depression',
    'Suicidal',
    'Anxiety',
    'Bipolar',
    'Stress',
    'Personality disorder'
]

df = df[df['status'].isin(valid_labels)]


: 

In [ ]:
# Split into features (x) and labels (y)
# x = cleaned text, y = mental health category
x= df['clean_sentences']
y = df['status']

: 

In [ ]:
# Verify the unique labels present in the target column
y.unique()

: 

In [ ]:
# Save the cleaned dataset to a CSV file for future use
df.to_csv(r"Cleaned Data.csv", index=False)

: 

---
## 7. Train / Test Split

In [ ]:
from sklearn.model_selection import train_test_split

# Split: 80% training, 20% testing
# random_state=42 ensures reproducibility
x_train, x_test, y_train, y_test = train_test_split(x,y, test_size=0.2, random_state=42)

: 

---
## 8. Tokenization & Sequence Padding

The LSTM model works with numbers, not raw text.
We convert each word to an integer ID, then pad all sequences to the same length (100).

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer

# Build the vocabulary from the training set only (to avoid data leakage)
# num_words=20000 keeps only the top 20,000 most frequent words
tokenizer = Tokenizer(num_words=20000)
tokenizer.fit_on_texts(x_train)


: 

In [ ]:
# Convert text to sequences of integer IDs
# Each word is replaced by its index in the vocabulary
x_train_seq = tokenizer.texts_to_sequences(x_train)
x_test_seq = tokenizer.texts_to_sequences(x_test)

: 

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Pad all sequences to length 100
# Shorter sentences are zero-padded at the start; longer ones are truncated
# maxlen=100 covers the vast majority of sentences in the dataset
x_train_pad = pad_sequences(x_train_seq, maxlen=100)
x_test_pad = pad_sequences(x_test_seq, maxlen=100)

: 

---
## 9. Label Encoding

The model outputs numbers, not strings.
LabelEncoder converts category names to integers (e.g. `'Anxiety'` → `0`).

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

# fit_transform on train: learns the mapping and encodes
# transform on test: uses the same mapping (no re-fitting to avoid leakage)
y_train_enc = label_encoder.fit_transform(y_train)
y_test_enc = label_encoder.transform(y_test)

: 

In [ ]:
# Verify the shape of the training data: (num_samples, maxlen)
x_train_pad.shape

: 

---
## 10. Build the LSTM Model

Architecture:
- **Embedding**: converts word IDs to dense 128-dimensional vectors
- **BatchNormalization + Dropout**: stabilize training and reduce overfitting
- **LSTM(128)**: first LSTM layer captures low-level sequential patterns
- **LSTM(64)**: second LSTM layer captures higher-level semantic patterns
- **Dense(7, softmax)**: output layer — one probability per class

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import *



: 

In [ ]:
model = Sequential()

# Embedding layer: maps each word index to a 128-dim dense vector
# input_dim=20000 matches the tokenizer vocabulary size
model.add(Embedding(input_dim=20000, output_dim=128, input_length=100))

# Normalize activations and apply dropout to prevent overfitting after embedding
model.add(BatchNormalization())
model.add(Dropout(0.3))

# First LSTM layer: return_sequences=True passes the full sequence to the next LSTM
model.add(LSTM(128, return_sequences=True))
  
model.add(Dropout(0.4))
model.add(BatchNormalization())

# Second LSTM layer: returns only the final hidden state
model.add(LSTM(64))

model.add(BatchNormalization())
model.add(Dropout(0.3))

# Output layer: 7 neurons (one per class) with softmax to produce probabilities
model.add(Dense(7, activation='softmax'))
  

: 

In [ ]:
# Print the model architecture summary
model.summary()

: 

---
## 11. Compile & Train the Model

In [ ]:
# Adam optimizer works well for NLP tasks
# sparse_categorical_crossentropy is used because labels are integers (not one-hot encoded)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Early stopping: stop training if val_accuracy doesn't improve for 1 epoch
# restore_best_weights=True ensures we keep the best version of the model
early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=1, restore_best_weights=True)

: 

In [ ]:
# Train the model
# batch_size=64: process 64 samples at a time
# epochs=10: max 10 passes through the data (early stopping may halt sooner)
history = model.fit(x_train_pad, y_train_enc, epochs=10, batch_size=64, validation_data=(x_test_pad, y_test_enc), callbacks=[early_stopping])

: 

---
## 12. Save the Model & Artifacts

Three files are needed for inference:
- `lstm_model.h5` — the trained model (weights + architecture)
- `tokenizer.pkl` — the fitted tokenizer (must match what was used during training)
- `label_encoder.pkl` — maps predicted integers back to label names

In [ ]:
import pickle

# Save the Keras model using TensorFlow's native format (preserves architecture + weights)
model.save("lstm_model.h5")

# Save the tokenizer and label encoder using pickle
# These are plain Python objects — pickle works correctly for them
pickle.dump(tokenizer, open('tokenizer.pkl', 'wb'))
pickle.dump(label_encoder, open('label_encoder.pkl', 'wb'))

: 

---
## 13. Load & Verify the Saved Artifacts

Always verify the saved model loads correctly before deploying.

In [ ]:
from tensorflow.keras.models import load_model

# Reload everything from disk to confirm the save was successful
model = load_model("lstm_model.h5")
tokenizer = pickle.load(open('tokenizer.pkl', 'rb'))
label_encoder = pickle.load(open('label_encoder.pkl', 'rb'))

: 